# Krishna River Basin — CH₄ Prediction: Complete Pipeline
## Replicating Paper Results + Fuzzy-DL Extension

**Paper:** *Explainable AI for Environmental Bioinformatics: Interpreting Methane (CH4) Prediction in the Krishna River Basin Using PDP, ALE, SHAP, and GAM*

### 📋 How to Use in Google Colab
1. Open [colab.research.google.com](https://colab.research.google.com)
2. Upload this notebook: **File → Upload notebook**
3. Run **Cell 1** first (installs packages), then **restart runtime**
4. In **Cell 3**, click the upload button and select `KR_basin.csv`
5. Run all remaining cells **in order** (Runtime → Run all)

---
| Section | Cells | Description |
|---------|-------|-------------|
| Paper Replication | 1–16 | ML models, PDP, ALE, SHAP, GAM |
| Future Extension | 18–23 | Fuzzy Logic + Deep Learning |


In [ ]:
# ============================================================================
# KRISHNA RIVER BASIN - CH4 PREDICTION: COMPLETE PIPELINE
# Replicating Paper Results + Fuzzy-DL Extension Framework
# Compatible with Google Colab
# ============================================================================
# 
# PAPER: "Explainable AI for Environmental Bioinformatics: Interpreting
#         Methane (CH4) Prediction in the Krishna River Basin Using
#         PDP, ALE, SHAP, and GAM"
#
# HOW TO USE IN GOOGLE COLAB:
#   1. Go to https://colab.research.google.com
#   2. File → Upload notebook → upload this .ipynb (or paste cells)
#   3. Run CELL 1 first (install packages)
#   4. In CELL 3, upload your KR_basin.csv when prompted
#   5. Run all cells in order
# ============================================================================

## Cell 1 — INSTALL PACKAGES  (Run this FIRST, restart runtime after)

In [ ]:
!pip install -q scikit-fuzzy tensorflow shap alibi pygam scikit-learn \
              pandas numpy matplotlib seaborn xgboost lightgbm

print("✅ All packages installed. If prompted, RESTART RUNTIME before continuing.")

## Cell 2 — IMPORT LIBRARIES

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.feature_selection import mutual_info_regression
from sklearn.inspection import partial_dependence, PartialDependenceDisplay

# Boosting
import xgboost as xgb
import lightgbm as lgb

# Explainability
import shap

# GAM
from pygam import LinearGAM, s

# Fuzzy Logic (for Part 2 — Future Extension)
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# Deep Learning (for Part 2 — Future Extension)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

print(f"✅ TensorFlow: {tf.__version__}")
print(f"✅ NumPy: {np.__version__}")
print(f"✅ Pandas: {pd.__version__}")
print("✅ All libraries loaded!")

## Cell 3 — UPLOAD DATA

In [ ]:
# 📁 FILE UPLOAD INSTRUCTION:
#    When you run this cell, a "Choose Files" button will appear.
#    Click it and upload your file: KR_basin.csv
#    The code will automatically detect and load it.
# ============================================================================

from google.colab import files

print("📂 Please upload your 'KR_basin.csv' file now...")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(filename)

print(f"\n✅ File loaded: '{filename}'")
print(f"   Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
print(f"\nColumn names in your file:")
for i, col in enumerate(df_raw.columns):
    print(f"  [{i}] {col}")

## Cell 4 — DEFINE FEATURE GROUPS

In [ ]:
# ⚠️  If your column names differ from below, update them to match
#     the column names printed in CELL 3.
# ============================================================================

# --- TARGET ---
TARGET = 'CH4 (µM) Average_Ambient Air'

# --- PROXIMATE (local) FEATURES ---
proximate_features = [
    'DO (mg L-1)',          # Dissolved Oxygen  — most important (~74.7%)
    'TP (µM)',              # Total Phosphorus  — 2nd (~9.1%)
    'SUVA 254',             # SUVA254           — 3rd (~7.7%)
    'DOC (mg L-1)',
    'CDOM 440 nm_simple (m-1)',
    'Water Temperatre (°C)',
    'pH',
    'Conductivity',
    'Turbidity (NTU)',
    'Nitrate (NO3) (mg L-1) (AVG)',
]

# --- DISTAL (catchment-scale) FEATURES ---
distal_features = [
    'Agriculture (%)',      # ~4.6% importance
    'Terrein Slope (%)',    # ~3.9% importance
    'Stream order',
    'Altitude (m)',
    'Waterbody (%)',
    'Forest (%)',
    'Flooded Vegetation (%)',
    'Built-up Area (%)',
    'Bareground (%)',
    'Rangeland (%)',
]

ALL_FEATURES = proximate_features + distal_features

# Verify columns
missing = [c for c in ALL_FEATURES + [TARGET] if c not in df_raw.columns]
if missing:
    print(f"⚠️  Missing columns: {missing}")
    print("   → Check CELL 3 output and update column names above.")
else:
    print("✅ All columns verified!")

# Extract X, y
X_full = df_raw[ALL_FEATURES].copy().fillna(df_raw[ALL_FEATURES].median())
y = df_raw[TARGET].copy().fillna(df_raw[TARGET].median())

print(f"\nDataset summary:")
print(f"  X shape : {X_full.shape}")
print(f"  y range : {y.min():.2f} – {y.max():.2f} µM CH4")
print(f"  y mean  : {y.mean():.2f} µM")

## Cell 5 — FEATURE SELECTION (replicating paper 2-step process)

In [ ]:
print("=" * 60)
print("STEP 1: Mutual Information Feature Selection")
print("=" * 60)

# Log-transform for normality / heteroscedasticity (as in paper)
X_log = np.log1p(X_full.clip(lower=0))
y_log = np.log1p(y.clip(lower=0))

mi_scores = mutual_info_regression(X_log, y_log, random_state=42)
mi_df = pd.DataFrame({'Feature': ALL_FEATURES, 'MI Score': mi_scores})
mi_df = mi_df.sort_values('MI Score', ascending=False)

print(mi_df.to_string(index=False))

# Correlation-based deduplication (remove >90% correlated pairs)
corr_matrix = X_log.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = [col for col in upper.columns if any(upper[col] > 0.90)]
print(f"\n⚠️  Highly correlated features (dropped): {to_drop_corr}")

# Final selected features matching paper (top 5 used in paper)
PAPER_FEATURES = ['DO (mg L-1)', 'TP (µM)', 'SUVA 254', 'Agriculture (%)', 'Terrein Slope (%)']
X = X_log[PAPER_FEATURES].copy()
y_model = y_log.copy()

print(f"\n✅ Final features for modeling: {PAPER_FEATURES}")

## Cell 6 — TRAIN/TEST SPLIT (80:20 as in paper)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_model, test_size=0.20, random_state=42
)

print(f"Train: {X_train.shape[0]} samples")
print(f"Test : {X_test.shape[0]} samples")

## Cell 7 — TRAIN ALL 6 ML MODELS (paper Table results)

In [ ]:
models_dict = {
    'Random Forest':      RandomForestRegressor(n_estimators=200, max_depth=None,
                                                 random_state=42, n_jobs=-1),
    'Linear Regression':  LinearRegression(),
    'Gradient Boosting':  GradientBoostingRegressor(n_estimators=200, random_state=42),
    'XGBoost':            xgb.XGBRegressor(n_estimators=200, random_state=42,
                                            verbosity=0),
    'LightGBM':           lgb.LGBMRegressor(n_estimators=200, random_state=42,
                                              verbose=-1),
    'Decision Tree':      DecisionTreeRegressor(random_state=42),
}

results = {}
for name, model in models_dict.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae  = mean_absolute_error(y_test, y_pred)
    mse  = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_test, y_pred)
    results[name] = {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2, 'model': model}
    print(f"{name:<20} | MAE={mae:.3f}  MSE={mse:.3f}  RMSE={rmse:.3f}  R²={r2:.2f}")

# PAPER EXPECTED (log-scale):
# RF: MAE=0.688, MSE=0.896, RMSE=0.946, R²=0.76  ← best
# XGB: R²=0.66,  LGBM: R²=0.63
# DT:  R²=0.59,  LR:   R²=0.53

## Cell 8 — FIGURE 6: ML MODEL COMPARISON BAR CHART

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

model_names = list(results.keys())
metrics     = ['MAE', 'MSE', 'RMSE', 'R2']
colors      = ['#2196F3', '#FF9800', '#4CAF50', '#F44336']
x           = np.arange(len(model_names))
width       = 0.2

for i, (metric, color) in enumerate(zip(metrics, colors)):
    vals = [results[m][metric] for m in model_names]
    bars = ax.bar(x + i * width, vals, width, label=metric, color=color, edgecolor='black', alpha=0.85)

ax.set_xlabel('ML Model', fontsize=13)
ax.set_ylabel('Score', fontsize=13)
ax.set_title('Figure 6: Model Comparison Across Different ML Algorithms', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(model_names, rotation=15, ha='right')
ax.legend(title='Metric', fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig6_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figure 6 saved: fig6_model_comparison.png")

## Cell 9 — 5-FOLD CROSS-VALIDATION (RF — best model)

In [ ]:
rf_best = results['Random Forest']['model']
cv_scores = cross_val_score(
    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    X, y_model, cv=5, scoring='r2'
)

print("Random Forest — 5-Fold Cross Validation:")
print(f"  R² per fold : {cv_scores.round(3)}")
print(f"  Mean R²     : {cv_scores.mean():.2f} ± {cv_scores.std():.2f}")
# Paper reports: 0.75 ± 0.13

## Cell 10 — FIGURE 1 & 3: FEATURE IMPORTANCE (RF)

In [ ]:
importances = rf_best.feature_importances_ * 100  # as percentage
feat_imp_df = pd.DataFrame({'Feature': PAPER_FEATURES, 'Importance (%)': importances})
feat_imp_df = feat_imp_df.sort_values('Importance (%)', ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(feat_imp_df['Feature'], feat_imp_df['Importance (%)'],
              color=['#1565C0', '#1976D2', '#1E88E5', '#42A5F5', '#90CAF9'],
              edgecolor='black')

for bar, val in zip(bars, feat_imp_df['Importance (%)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xlabel('Feature', fontsize=12)
ax.set_ylabel('Feature Importance (%)', fontsize=12)
ax.set_title('Figure 1 & 3: Feature Importance (Random Forest)\nCH₄ Concentration Prediction',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, max(importances) * 1.2)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig('fig1_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figure 1/3 saved: fig1_feature_importance.png")
print("\nExpected from paper: DO≈74.7%, TP≈9.1%, SUVA254≈7.7%, Agriculture≈4.6%, Slope≈3.9%")

## Cell 11 — FIGURE 2: PARTIAL DEPENDENCE PLOTS (PDP)

In [ ]:
# PDP: marginal effect of each predictor, averaging over all others
# Paper finding: DO has NEGATIVE effect, TP has threshold ~3.5µM,
#                SUVA254 positive effect above ~1.2, Ag & Slope are flat/weak

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

feature_labels = {
    'DO (mg L-1)':      'DO (mg L⁻¹)',
    'SUVA 254':         'SUVA₂₅₄',
    'TP (µM)':          'TP (µM)',
    'Agriculture (%)':  'Agriculture (%)',
    'Terrein Slope (%)':'Terrain Slope (%)',
}

scatter_colors = plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(X_test)))

for idx, feat in enumerate(PAPER_FEATURES):
    ax = axes[idx]
    pdp_result = partial_dependence(rf_best, X_train, features=[idx], kind='average')
    values  = pdp_result['grid_values'][0]
    pdp_avg = pdp_result['average'][0]

    # Scatter of actual points
    ax.scatter(X_test[feat], y_test, alpha=0.5, s=40,
               c=y_test.values, cmap='RdYlGn', edgecolors='k', linewidth=0.3, zorder=2)

    # PDP line
    ax.plot(values, pdp_avg, color='steelblue', linewidth=2.5, zorder=3, label='PDP')

    ax.set_xlabel(feature_labels.get(feat, feat), fontsize=11)
    ax.set_ylabel('Partial Dependence', fontsize=11)
    ax.set_title(f'PDP: {feature_labels.get(feat, feat)}', fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=9)

axes[-1].axis('off')  # hide empty subplot
fig.suptitle('Figure 2: Partial Dependence Plots (PDP)\nMarginal Effect of Each Feature on CH₄',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig2_pdp.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figure 2 saved: fig2_pdp.png")

## Cell 12 — ALE PLOTS (Accumulated Local Effects)

In [ ]:
# ALE corrects for feature correlations — avoids extrapolation bias in PDPs
# Paper: DO strong negative, SUVA254 threshold at 1.0–1.5, TP sharp rise >4µM,
#         Agriculture and slope show nonlinear U-shape / inverted-U

def compute_ale(model, X_data, feature_idx, n_bins=20):
    """
    Simple ALE computation for regression.
    ALE measures local effect by conditioning on small bins of feature values.
    """
    feat_vals = X_data.iloc[:, feature_idx].values
    percentiles = np.percentile(feat_vals, np.linspace(0, 100, n_bins + 1))
    percentiles = np.unique(percentiles)

    ale_effects = []
    bin_centers = []

    for i in range(len(percentiles) - 1):
        lo, hi = percentiles[i], percentiles[i+1]
        mask = (feat_vals >= lo) & (feat_vals <= hi)
        if mask.sum() < 2:
            continue

        X_lo = X_data.copy()
        X_hi = X_data.copy()
        X_lo.iloc[:, feature_idx] = lo
        X_hi.iloc[:, feature_idx] = hi

        diff = model.predict(X_hi[mask]) - model.predict(X_lo[mask])
        ale_effects.append(diff.mean())
        bin_centers.append((lo + hi) / 2)

    ale_cumulative = np.cumsum(ale_effects)
    ale_centered = ale_cumulative - ale_cumulative.mean()
    return np.array(bin_centers), ale_centered


fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, feat in enumerate(PAPER_FEATURES):
    ax = axes[idx]
    bins, ale = compute_ale(rf_best, X_train, feature_idx=idx, n_bins=25)

    ax.scatter(X_test[feat], y_test, alpha=0.4, s=35,
               c=y_test.values, cmap='RdYlGn', edgecolors='k', linewidth=0.3, zorder=2)
    ax.plot(bins, ale, color='darkorange', linewidth=2.5, zorder=3, label='ALE')
    ax.axhline(0, color='gray', linestyle='--', linewidth=1)
    ax.set_xlabel(feature_labels.get(feat, feat), fontsize=11)
    ax.set_ylabel('ALE Effect', fontsize=11)
    ax.set_title(f'ALE: {feature_labels.get(feat, feat)}', fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=9)

axes[-1].axis('off')
fig.suptitle('Accumulated Local Effects (ALE)\nHandles Feature Correlations — More Reliable than PDP',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('ale_plots.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ ALE plots saved: ale_plots.png")

## Cell 13 — FIGURE 4: SHAP ANALYSIS

In [ ]:
# SHAP: Shapley values — both global + local explanations
# Paper: DO shows +2 to +4 SHAP when low, TP >3µM positive,
#         SUVA254 sharp positive after ~1.0–1.3

explainer = shap.TreeExplainer(rf_best)
shap_values = explainer.shap_values(X_test)

# --- 4a. SHAP Summary Beeswarm (global) ---
fig, ax = plt.subplots(figsize=(9, 6))
shap.summary_plot(shap_values, X_test, feature_names=PAPER_FEATURES,
                  show=False, plot_size=None)
plt.title('Figure 4a: SHAP Summary Plot — Feature Impact on CH₄ Prediction',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4a_shap_summary.png', dpi=300, bbox_inches='tight')
plt.show()

# --- 4b. SHAP Dependence plots (one per feature — replicates Fig 4 in paper) ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, feat in enumerate(PAPER_FEATURES):
    ax = axes[idx]
    shap.dependence_plot(
        feat, shap_values, X_test,
        feature_names=PAPER_FEATURES,
        ax=ax, show=False,
        interaction_index=None,
        dot_size=50, alpha=0.7
    )
    ax.set_title(f'SHAP: {feature_labels.get(feat, feat)}', fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3)

axes[-1].axis('off')
fig.suptitle('Figure 4: SHAP Dependence Plots — Local & Global Feature Effects',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig4b_shap_dependence.png', dpi=300, bbox_inches='tight')
plt.show()

# --- 4c. SHAP Bar chart (mean |SHAP|) ---
fig, ax = plt.subplots(figsize=(8, 5))
shap.summary_plot(shap_values, X_test, feature_names=PAPER_FEATURES,
                  plot_type='bar', show=False)
plt.title('SHAP Feature Importance (Mean |SHAP|)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4c_shap_bar.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ SHAP figures saved: fig4a, fig4b, fig4c")

## Cell 14 — FIGURE 5: GENERALIZED ADDITIVE MODELS (GAM)

In [ ]:
# GAM: smooth nonlinear shapes with confidence intervals
# Paper: SUVA254 threshold at 1.0–1.5, TP saturation >3.5µM, U-shape for slope

X_train_arr = X_train.values
X_test_arr  = X_test.values

# Fit GAM with splines for each feature
gam = LinearGAM(
    s(0, n_splines=10) +   # DO
    s(1, n_splines=10) +   # TP
    s(2, n_splines=10) +   # SUVA254
    s(3, n_splines=8)  +   # Agriculture
    s(4, n_splines=8)      # Slope
).fit(X_train_arr, y_train.values)

print(f"GAM Pseudo-R²: {gam.statistics_['pseudo_r2']['explained_deviance']:.3f}")

# Plot GAM partial effects
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, feat in enumerate(PAPER_FEATURES):
    ax = axes[idx]
    XX = gam.generate_X_grid(term=idx)
    pdep, confi = gam.partial_dependence(term=idx, X=XX, width=0.95)

    ax.scatter(X_test.iloc[:, idx], y_test, alpha=0.4, s=40,
               c=y_test.values, cmap='RdYlGn', edgecolors='k', linewidth=0.3, zorder=2)
    ax.plot(XX[:, idx], pdep, color='navy', linewidth=2.5, zorder=3, label='GAM smooth')
    ax.fill_between(XX[:, idx], confi[:, 0], confi[:, 1],
                    alpha=0.25, color='blue', label='95% CI')
    ax.set_xlabel(feature_labels.get(feat, feat), fontsize=11)
    ax.set_ylabel('Effect', fontsize=11)
    ax.set_title(f'GAM: {feature_labels.get(feat, feat)}', fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=9)

axes[-1].axis('off')
fig.suptitle('Figure 5: Generalized Additive Models (GAM)\nSmooth Effects with 95% Confidence Intervals',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig5_gam.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figure 5 saved: fig5_gam.png")

## Cell 15 — COMBINED INTERPRETABILITY COMPARISON (All 4 Methods)

In [ ]:
# Visualises PDP, ALE, SHAP, and GAM side-by-side for DO (most important)

fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=False)
feat_idx = 0  # DO — most important feature

# --- PDP ---
ax = axes[0]
pdp_result = partial_dependence(rf_best, X_train, features=[feat_idx], kind='average')
ax.plot(pdp_result['grid_values'][0], pdp_result['average'][0],
        color='steelblue', linewidth=2.5)
ax.scatter(X_test[PAPER_FEATURES[feat_idx]], y_test, alpha=0.4, s=30,
           c=y_test.values, cmap='RdYlGn', edgecolors='k', linewidth=0.3)
ax.set_title('PDP\n(Marginal Effect)', fontsize=12, fontweight='bold')
ax.set_xlabel('DO (mg L⁻¹)', fontsize=11); ax.set_ylabel('Effect on CH₄', fontsize=11)
ax.grid(alpha=0.3)

# --- ALE ---
ax = axes[1]
bins, ale = compute_ale(rf_best, X_train, feature_idx=feat_idx, n_bins=25)
ax.plot(bins, ale, color='darkorange', linewidth=2.5)
ax.scatter(X_test[PAPER_FEATURES[feat_idx]], y_test, alpha=0.4, s=30,
           c=y_test.values, cmap='RdYlGn', edgecolors='k', linewidth=0.3)
ax.axhline(0, color='gray', linestyle='--', linewidth=1)
ax.set_title('ALE\n(Local Effect)', fontsize=12, fontweight='bold')
ax.set_xlabel('DO (mg L⁻¹)', fontsize=11); ax.set_ylabel('ALE Effect', fontsize=11)
ax.grid(alpha=0.3)

# --- SHAP ---
ax = axes[2]
ax.scatter(X_test[PAPER_FEATURES[feat_idx]], shap_values[:, feat_idx],
           c=X_test[PAPER_FEATURES[feat_idx]], cmap='RdYlGn',
           s=50, edgecolors='k', linewidth=0.3, alpha=0.8)
# Smooth trend line
do_sorted = np.sort(X_test[PAPER_FEATURES[feat_idx]].values)
ax.set_title('SHAP\n(Shapley Values)', fontsize=12, fontweight='bold')
ax.set_xlabel('DO (mg L⁻¹)', fontsize=11); ax.set_ylabel('SHAP Value', fontsize=11)
ax.axhline(0, color='gray', linestyle='--', linewidth=1)
ax.grid(alpha=0.3)

# --- GAM ---
ax = axes[3]
XX = gam.generate_X_grid(term=feat_idx)
pdep, confi = gam.partial_dependence(term=feat_idx, X=XX, width=0.95)
ax.plot(XX[:, feat_idx], pdep, color='navy', linewidth=2.5)
ax.fill_between(XX[:, feat_idx], confi[:, 0], confi[:, 1], alpha=0.25, color='blue')
ax.scatter(X_test[PAPER_FEATURES[feat_idx]], y_test, alpha=0.4, s=30,
           c=y_test.values, cmap='RdYlGn', edgecolors='k', linewidth=0.3)
ax.set_title('GAM\n(Smooth + CI)', fontsize=12, fontweight='bold')
ax.set_xlabel('DO (mg L⁻¹)', fontsize=11); ax.set_ylabel('Partial Effect', fontsize=11)
ax.grid(alpha=0.3)

fig.suptitle('All 4 Interpretability Methods — DO (Dissolved Oxygen) vs CH₄\n'
             'All agree: negative relationship — low DO → high CH₄',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('comparison_all_methods.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Combined comparison saved: comparison_all_methods.png")

## Cell 16 — FINAL MODEL PERFORMANCE SUMMARY TABLE

In [ ]:
print("\n" + "=" * 75)
print(" FINAL MODEL PERFORMANCE SUMMARY (replicating paper Table)")
print("=" * 75)
print(f"{'Model':<22} {'MAE':>8} {'MSE':>8} {'RMSE':>8} {'R²':>6}")
print("-" * 75)
for name, res in results.items():
    marker = " ← BEST" if name == 'Random Forest' else ""
    print(f"{name:<22} {res['MAE']:>8.3f} {res['MSE']:>8.3f} {res['RMSE']:>8.3f} {res['R2']:>6.2f}{marker}")
print("=" * 75)
print("\nPaper reports (RF): MAE=0.688, MSE=0.896, RMSE=0.946, R²=0.76")
print("Cross-validation:   R² = 0.75 ± 0.13 (paper value)")

cv_rf = cross_val_score(
    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    X, y_model, cv=5, scoring='r2'
)
print(f"Your CV result:     R² = {cv_rf.mean():.2f} ± {cv_rf.std():.2f}")

## Cell 17 — DOWNLOAD ALL FIGURES

In [ ]:
import zipfile, os

fig_files = [
    'fig1_feature_importance.png',
    'fig2_pdp.png',
    'ale_plots.png',
    'fig4a_shap_summary.png',
    'fig4b_shap_dependence.png',
    'fig4c_shap_bar.png',
    'fig5_gam.png',
    'fig6_model_comparison.png',
    'comparison_all_methods.png',
]

with zipfile.ZipFile('KR_basin_paper_figures.zip', 'w') as zf:
    for f in fig_files:
        if os.path.exists(f):
            zf.write(f)

files.download('KR_basin_paper_figures.zip')
print("✅ All figures downloaded!")


# ============================================================================
# ============================================================================
#
#  PART 2 — FUTURE EXTENSION: FUZZY LOGIC + DEEP LEARNING (Next Month)
#
# ============================================================================
# ============================================================================

## Cell 18 — FUZZY LOGIC FEATURE ENGINEERING

In [ ]:
# Builds interpretable fuzzy features from DO, TP, SUVA254
# These fuzzy features are then fed into the Deep Learning model

class FuzzyFeatureEngine:
    """
    Converts raw environmental measurements into fuzzy linguistic features.
    Captures expert knowledge about methanogenesis conditions.
    """

    def __init__(self):
        self.systems = {}
        self._build_systems()

    def _build_systems(self):

        # ── SYSTEM 1: Redox Potential (from DO) ──────────────────────────────
        do_u = np.arange(0, 15, 0.1)
        rp_u = np.arange(0, 1, 0.01)

        do_ant = ctrl.Antecedent(do_u, 'dissolved_oxygen')
        rp_con = ctrl.Consequent(rp_u, 'redox_potential')

        do_ant['anoxic']       = fuzz.trapmf(do_u, [0,   0,   1,   2  ])
        do_ant['hypoxic']      = fuzz.trimf( do_u, [1,   3,   5      ])
        do_ant['normal']       = fuzz.trimf( do_u, [4,   7,   10     ])
        do_ant['supersatd']    = fuzz.trapmf(do_u, [9,   11,  15, 15 ])

        rp_con['very_high']    = fuzz.trapmf(rp_u, [0, 0,   0.2, 0.4])
        rp_con['high']         = fuzz.trimf( rp_u, [0.3, 0.5, 0.7   ])
        rp_con['low']          = fuzz.trimf( rp_u, [0.6, 0.8, 0.9   ])
        rp_con['very_low']     = fuzz.trapmf(rp_u, [0.85,0.95, 1, 1 ])

        self.systems['redox'] = ctrl.ControlSystem([
            ctrl.Rule(do_ant['anoxic'],    rp_con['very_high']),
            ctrl.Rule(do_ant['hypoxic'],   rp_con['high']),
            ctrl.Rule(do_ant['normal'],    rp_con['low']),
            ctrl.Rule(do_ant['supersatd'], rp_con['very_low']),
        ])

        # ── SYSTEM 2: Nutrient Productivity (from TP) ────────────────────────
        tp_u  = np.arange(0, 200, 1)
        pr_u  = np.arange(0, 1, 0.01)

        tp_ant = ctrl.Antecedent(tp_u, 'total_phosphorus')
        pr_con = ctrl.Consequent(pr_u, 'productivity')

        tp_ant['oligo']  = fuzz.trapmf(tp_u, [0,   0,   10,  30 ])
        tp_ant['meso']   = fuzz.trimf( tp_u, [20,  50,  80      ])
        tp_ant['eutro']  = fuzz.trimf( tp_u, [70,  120, 160     ])
        tp_ant['hyper']  = fuzz.trapmf(tp_u, [150, 180, 200, 200])

        pr_con['low']      = fuzz.trapmf(pr_u, [0, 0,  0.2, 0.4])
        pr_con['moderate'] = fuzz.trimf( pr_u, [0.3, 0.5, 0.7  ])
        pr_con['high']     = fuzz.trimf( pr_u, [0.6, 0.8, 0.9  ])
        pr_con['very_high']= fuzz.trapmf(pr_u, [0.85,0.95, 1, 1])

        self.systems['nutrient'] = ctrl.ControlSystem([
            ctrl.Rule(tp_ant['oligo'], pr_con['low']),
            ctrl.Rule(tp_ant['meso'],  pr_con['moderate']),
            ctrl.Rule(tp_ant['eutro'], pr_con['high']),
            ctrl.Rule(tp_ant['hyper'], pr_con['very_high']),
        ])

        # ── SYSTEM 3: Organic Matter Quality (from SUVA254) ──────────────────
        sv_u  = np.arange(0, 10, 0.1)
        om_u  = np.arange(0, 1, 0.01)

        sv_ant = ctrl.Antecedent(sv_u, 'suva254')
        om_con = ctrl.Consequent(om_u, 'om_quality')

        sv_ant['low_arom']  = fuzz.trapmf(sv_u, [0, 0, 1, 2  ])
        sv_ant['moderate']  = fuzz.trimf( sv_u, [1.5, 3, 4.5 ])
        sv_ant['high_arom'] = fuzz.trapmf(sv_u, [4,  6, 10,10])

        om_con['labile']        = fuzz.trapmf(om_u, [0, 0, 0.3, 0.5])
        om_con['intermediate']  = fuzz.trimf( om_u, [0.4, 0.6, 0.8 ])
        om_con['recalcitrant']  = fuzz.trapmf(om_u, [0.7, 0.9, 1, 1])

        self.systems['organic'] = ctrl.ControlSystem([
            ctrl.Rule(sv_ant['low_arom'],  om_con['labile']),
            ctrl.Rule(sv_ant['moderate'],  om_con['intermediate']),
            ctrl.Rule(sv_ant['high_arom'], om_con['recalcitrant']),
        ])

    def transform(self, X_df):
        """Compute fuzzy features for a DataFrame."""
        rp_vals, pr_vals, om_vals = [], [], []

        rp_sim = ctrl.ControlSystemSimulation(self.systems['redox'])
        nu_sim = ctrl.ControlSystemSimulation(self.systems['nutrient'])
        om_sim = ctrl.ControlSystemSimulation(self.systems['organic'])

        for _, row in X_df.iterrows():
            # Redox from DO
            try:
                rp_sim.input['dissolved_oxygen'] = np.clip(row['DO (mg L-1)'], 0, 14.9)
                rp_sim.compute()
                rp_vals.append(rp_sim.output['redox_potential'])
            except:
                rp_vals.append(0.5)

            # Productivity from TP
            try:
                nu_sim.input['total_phosphorus'] = np.clip(row['TP (µM)'], 0, 199)
                nu_sim.compute()
                pr_vals.append(nu_sim.output['productivity'])
            except:
                pr_vals.append(0.5)

            # OM quality from SUVA254
            try:
                om_sim.input['suva254'] = np.clip(row['SUVA 254'], 0, 9.9)
                om_sim.compute()
                om_vals.append(om_sim.output['om_quality'])
            except:
                om_vals.append(0.5)

        fuzzy_df = pd.DataFrame({
            'fuzzy_redox':      rp_vals,
            'fuzzy_productivity': pr_vals,
            'fuzzy_om_quality': om_vals,
        }, index=X_df.index)

        # Composite methane potential index
        fuzzy_df['fuzzy_CH4_index'] = (
            fuzzy_df['fuzzy_redox']       * 0.50 +
            fuzzy_df['fuzzy_productivity'] * 0.30 +
            fuzzy_df['fuzzy_om_quality']  * 0.20
        )

        return fuzzy_df


# Compute fuzzy features (uses X_full — unlogged, original scale)
print("Computing fuzzy features... (may take ~1–2 min)")
fuzzy_engine = FuzzyFeatureEngine()
X_fuzzy = fuzzy_engine.transform(X_full)
print(f"✅ Fuzzy features computed: {X_fuzzy.shape}")
print(X_fuzzy.describe().round(3))

## Cell 19 — FUZZY MEMBERSHIP FUNCTION VISUALIZATION

In [ ]:
def plot_fuzzy_memberships():
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # DO
    ax = axes[0]
    do = np.arange(0, 15, 0.1)
    ax.plot(do, fuzz.trapmf(do, [0,  0,  1,  2 ]), lw=2, label='Anoxic')
    ax.plot(do, fuzz.trimf( do, [1,  3,  5    ]), lw=2, label='Hypoxic')
    ax.plot(do, fuzz.trimf( do, [4,  7,  10   ]), lw=2, label='Normal')
    ax.plot(do, fuzz.trapmf(do, [9, 11, 15, 15]), lw=2, label='Supersaturated')
    ax.set_xlabel('DO (mg/L)', fontsize=12)
    ax.set_ylabel('Membership Degree', fontsize=12)
    ax.set_title('Fuzzy Sets: Dissolved Oxygen', fontsize=13, fontweight='bold')
    ax.legend(); ax.grid(alpha=0.3)

    # TP
    ax = axes[1]
    tp = np.arange(0, 200, 1)
    ax.plot(tp, fuzz.trapmf(tp, [0,   0,   10,  30 ]), lw=2, label='Oligotrophic')
    ax.plot(tp, fuzz.trimf( tp, [20,  50,  80      ]), lw=2, label='Mesotrophic')
    ax.plot(tp, fuzz.trimf( tp, [70,  120, 160     ]), lw=2, label='Eutrophic')
    ax.plot(tp, fuzz.trapmf(tp, [150, 180, 200, 200]), lw=2, label='Hypereutrophic')
    ax.set_xlabel('Total Phosphorus (µM)', fontsize=12)
    ax.set_ylabel('Membership Degree', fontsize=12)
    ax.set_title('Fuzzy Sets: Total Phosphorus', fontsize=13, fontweight='bold')
    ax.legend(); ax.grid(alpha=0.3)

    # SUVA254
    ax = axes[2]
    sv = np.arange(0, 10, 0.1)
    ax.plot(sv, fuzz.trapmf(sv, [0, 0, 1,  2 ]), lw=2, label='Low Aromatic')
    ax.plot(sv, fuzz.trimf( sv, [1.5, 3, 4.5 ]), lw=2, label='Moderate')
    ax.plot(sv, fuzz.trapmf(sv, [4,  6, 10,10]), lw=2, label='High Aromatic')
    ax.set_xlabel('SUVA₂₅₄', fontsize=12)
    ax.set_ylabel('Membership Degree', fontsize=12)
    ax.set_title('Fuzzy Sets: SUVA₂₅₄', fontsize=13, fontweight='bold')
    ax.legend(); ax.grid(alpha=0.3)

    plt.suptitle('Fuzzy Logic Membership Functions\nEncoding Expert Knowledge of Methanogenesis',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('fuzzy_membership_functions.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✅ Fuzzy membership functions saved")

plot_fuzzy_memberships()

## Cell 20 — DEEP LEARNING MODEL (Fuzzy + Raw Features Combined)

In [ ]:
# Combine raw (log-transformed) + fuzzy features
X_combined = pd.concat([X, X_fuzzy.reset_index(drop=True)], axis=1)
X_combined_arr = X_combined.values
y_arr = y_model.values

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_combined_arr)

# Split
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y_arr,
                                            test_size=0.20, random_state=42)

# Build deep learning model
def build_dl_model(input_dim):
    inputs = keras.Input(shape=(input_dim,), name='input')
    x = layers.Dense(128, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dense(16, activation='relu')(x)
    output = layers.Dense(1, name='ch4_output')(x)
    model = keras.Model(inputs, output)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='huber',
        metrics=['mae']
    )
    return model

dl_model = build_dl_model(X_tr.shape[1])
dl_model.summary()

# Callbacks
cb = [
    callbacks.EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6),
]

history = dl_model.fit(
    X_tr, y_tr,
    validation_data=(X_te, y_te),
    epochs=300,
    batch_size=16,
    callbacks=cb,
    verbose=1
)

# Evaluate
y_dl_pred = dl_model.predict(X_te).flatten()
dl_mae    = mean_absolute_error(y_te, y_dl_pred)
dl_rmse   = np.sqrt(mean_squared_error(y_te, y_dl_pred))
dl_r2     = r2_score(y_te, y_dl_pred)

print(f"\n✅ Fuzzy-DL Model Results:")
print(f"   MAE  = {dl_mae:.3f}")
print(f"   RMSE = {dl_rmse:.3f}")
print(f"   R²   = {dl_r2:.3f}")
print(f"\n   RF (paper baseline) R² = {results['Random Forest']['R2']:.3f}")
print(f"   Improvement: {(dl_r2 - results['Random Forest']['R2'])*100:+.1f}%")

## Cell 21 — FUZZY-DL RESULTS VISUALIZATION

In [ ]:
fig = plt.figure(figsize=(18, 10))

# Training history
ax1 = fig.add_subplot(2, 3, 1)
ax1.plot(history.history['loss'],     label='Train Loss', lw=2)
ax1.plot(history.history['val_loss'], label='Val Loss',   lw=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Huber Loss')
ax1.set_title('Training History', fontweight='bold')
ax1.legend(); ax1.grid(alpha=0.3)

# MAE history
ax2 = fig.add_subplot(2, 3, 2)
ax2.plot(history.history['mae'],     label='Train MAE', lw=2)
ax2.plot(history.history['val_mae'], label='Val MAE',   lw=2)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('MAE (µM)')
ax2.set_title('MAE History', fontweight='bold')
ax2.legend(); ax2.grid(alpha=0.3)

# Predicted vs Actual
ax3 = fig.add_subplot(2, 3, 3)
ax3.scatter(y_te, y_dl_pred, alpha=0.7, s=50, edgecolors='k', linewidth=0.5)
mn, mx = min(y_te.min(), y_dl_pred.min()), max(y_te.max(), y_dl_pred.max())
ax3.plot([mn, mx], [mn, mx], 'r--', lw=2, label='1:1 line')
ax3.set_xlabel('Actual CH₄ (log µM)'); ax3.set_ylabel('Predicted CH₄ (log µM)')
ax3.set_title(f'Predicted vs Actual (R²={dl_r2:.3f})', fontweight='bold')
ax3.legend(); ax3.grid(alpha=0.3)

# Residuals
ax4 = fig.add_subplot(2, 3, 4)
resid = y_te - y_dl_pred
ax4.scatter(y_dl_pred, resid, alpha=0.6, s=50, edgecolors='k', linewidth=0.5)
ax4.axhline(0, color='r', linestyle='--', lw=2)
ax4.set_xlabel('Predicted'); ax4.set_ylabel('Residual')
ax4.set_title('Residual Plot', fontweight='bold')
ax4.grid(alpha=0.3)

# Residual histogram
ax5 = fig.add_subplot(2, 3, 5)
ax5.hist(resid, bins=25, edgecolor='black', alpha=0.75, color='steelblue')
ax5.axvline(0, color='r', linestyle='--', lw=2)
ax5.set_xlabel('Residual'); ax5.set_ylabel('Count')
ax5.set_title('Residual Distribution', fontweight='bold')
ax5.grid(alpha=0.3)

# Model comparison
ax6 = fig.add_subplot(2, 3, 6)
model_names_cmp = list(results.keys()) + ['Fuzzy-DL']
r2_vals = [results[m]['R2'] for m in results] + [dl_r2]
colors_cmp = ['#90CAF9'] * len(results) + ['#FF7043']
bars = ax6.bar(model_names_cmp, r2_vals, color=colors_cmp, edgecolor='black', alpha=0.85)
ax6.set_ylabel('R²'); ax6.set_ylim(0, 1.05)
ax6.set_title('Model R² Comparison', fontweight='bold')
plt.xticks(rotation=30, ha='right')
ax6.grid(axis='y', alpha=0.3)
for bar, v in zip(bars, r2_vals):
    ax6.text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.2f}',
             ha='center', fontsize=9, fontweight='bold')

fig.suptitle('Fuzzy-DL Framework Results — Krishna River Basin CH₄ Prediction',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fuzzy_dl_results.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Fuzzy-DL results saved: fuzzy_dl_results.png")

## Cell 22 — SHAP ON FUZZY-DL (Explainability for DL)

In [ ]:
feature_names_combined = PAPER_FEATURES + list(X_fuzzy.columns)

# Use a background subset for efficiency
background = X_tr[:50]
dl_explainer = shap.GradientExplainer(dl_model, background)
shap_dl_vals = dl_explainer.shap_values(X_te[:30])

if isinstance(shap_dl_vals, list):
    shap_dl_vals = shap_dl_vals[0]
shap_dl_vals = shap_dl_vals.squeeze()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Summary
plt.sca(axes[0])
shap.summary_plot(shap_dl_vals, X_te[:30],
                  feature_names=feature_names_combined,
                  show=False, plot_size=None)
axes[0].set_title('SHAP — Fuzzy-DL Model', fontsize=13, fontweight='bold')

# Bar
plt.sca(axes[1])
shap.summary_plot(shap_dl_vals, X_te[:30],
                  feature_names=feature_names_combined,
                  plot_type='bar', show=False)
axes[1].set_title('Feature Importance — Fuzzy-DL', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('shap_fuzzy_dl.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ SHAP on Fuzzy-DL saved: shap_fuzzy_dl.png")

## Cell 23 — DOWNLOAD ALL OUTPUTS

In [ ]:
import zipfile, os

all_outputs = [
    'fig1_feature_importance.png',
    'fig2_pdp.png',
    'ale_plots.png',
    'fig4a_shap_summary.png',
    'fig4b_shap_dependence.png',
    'fig4c_shap_bar.png',
    'fig5_gam.png',
    'fig6_model_comparison.png',
    'comparison_all_methods.png',
    'fuzzy_membership_functions.png',
    'fuzzy_dl_results.png',
    'shap_fuzzy_dl.png',
]

with zipfile.ZipFile('KR_basin_ALL_results.zip', 'w') as zf:
    for f in all_outputs:
        if os.path.exists(f):
            zf.write(f)
            print(f"  Added: {f}")

files.download('KR_basin_ALL_results.zip')
print("\n✅ Complete results downloaded!")

print("""
╔══════════════════════════════════════════════════════════════╗
║          WHAT EACH CELL PRODUCES (Quick Reference)          ║
╠══════════════════════════════════════════════════════════════╣
║  CELL  7 → Model training (RF, LR, GB, XGB, LGBM, DT)      ║
║  CELL  8 → Figure 6: Model comparison bar chart             ║
║  CELL  9 → 5-fold CV (RF, target R²=0.75±0.13)             ║
║  CELL 10 → Figure 1/3: Feature importance bar chart         ║
║  CELL 11 → Figure 2: Partial Dependence Plots (PDP)         ║
║  CELL 12 → ALE plots (handles feature correlation)          ║
║  CELL 13 → Figure 4: SHAP (summary + dependence + bar)      ║
║  CELL 14 → Figure 5: GAM with 95% CIs                       ║
║  CELL 15 → All 4 methods side-by-side for DO                ║
║  ─────────── FUTURE WORK (next month) ───────────────────── ║
║  CELL 18 → Fuzzy Feature Engineering (DO, TP, SUVA254)      ║
║  CELL 19 → Fuzzy membership function plots                   ║
║  CELL 20 → Hybrid Fuzzy-DL training                         ║
║  CELL 21 → Fuzzy-DL result visualizations                   ║
║  CELL 22 → SHAP on Fuzzy-DL                                 ║
╚══════════════════════════════════════════════════════════════╝
""")